# SetFit-CEFR — train on Google Colab

**You edit one cell** (the next one). Every other cell is boilerplate you shouldn't need to touch.

The 3 things you set:

| Var          | Meaning                                                                  |
|--------------|--------------------------------------------------------------------------|
| `DATA_DIR`   | Folder on your Drive that contains the 5 CSVs (names listed below).      |
| `OUTPUT_DIR` | Folder on your Drive where `models/` and `predictions/` are written.     |
| `MODELS`     | Backbones to train, one after another. See `docs/models.md` for choices. |

Every other hyperparameter (epochs, batch size, `sample_per_class`, seed, eval split, label order, …) comes from `configs/default.yaml`. If you want to change any of those, edit that file — the notebook will pick it up on the next run.

**Expected filenames in `DATA_DIR`** (columns: `writing_id,l1,cefr_level,text`):

`norm-EFCAMDAT-train.csv` · `norm-EFCAMDAT-remainder.csv` · `norm-EFCAMDAT-test.csv` · `norm-KUPA-KEYS.csv` · `norm-CELVA-SP.csv`

> **Colab**: Runtime → Change runtime type → T4 GPU. Then **Runtime → Run all**.

## The only cell you edit

In [ ]:
# ============================================================================
#  USER CONFIG — edit the three values below, then run every cell.
# ============================================================================

DATA_DIR   = "/content/drive/MyDrive/cefr-csvs"       # <-- 5 CSVs live here
OUTPUT_DIR = "/content/drive/MyDrive/cefr-outputs"    # <-- models + predictions land here

MODELS = [
    "sentence-transformers/paraphrase-MiniLM-L6-v2",            # small, fast sanity check
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",  # strong multilingual default
]

# ============================================================================

## Setup (you shouldn't need to touch anything below)

In [ ]:
# Clone or update the repo, then cd into it.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/berstearns/setfit-cefr.git"
REPO_DIR = Path("setfit-cefr")

if Path("pyproject.toml").exists() and Path("src/setfit_cefr").exists():
    print("Already in the setfit-cefr repo root.")
else:
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    os.chdir(REPO_DIR)
print("CWD:", Path().resolve())

In [ ]:
# Install the package into this kernel. ~2 min the first time.
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
import setfit, setfit_cefr  # noqa: E402
print(f"setfit {setfit.__version__} · setfit_cefr {setfit_cefr.__version__}")

In [ ]:
# Mount Google Drive on Colab. No-op elsewhere.
try:
    from google.colab import drive  # type: ignore
except ImportError:
    print("Not on Colab — skipping Drive mount.")
else:
    drive.mount("/content/drive")

In [ ]:
# Validate DATA_DIR and link the 5 CSVs into ./data/ (the path configs/default.yaml expects).
import shutil
from pathlib import Path

import pandas as pd

EXPECTED = [
    "norm-EFCAMDAT-train.csv",
    "norm-EFCAMDAT-remainder.csv",
    "norm-EFCAMDAT-test.csv",
    "norm-KUPA-KEYS.csv",
    "norm-CELVA-SP.csv",
]

src_dir = Path(DATA_DIR).expanduser()
if not src_dir.is_dir():
    raise FileNotFoundError(f"DATA_DIR does not exist: {src_dir}")

missing = [f for f in EXPECTED if not (src_dir / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing from {src_dir}: {missing}\n"
        f"Rename your files to match, or edit configs/default.yaml + the EXPECTED list above."
    )

Path("data").mkdir(exist_ok=True)
for fname in EXPECTED:
    link = Path("data") / fname
    if link.exists() or link.is_symlink():
        link.unlink()
    try:
        link.symlink_to(src_dir / fname)
    except OSError:
        # Drive-backed paths don't support symlinks — copy instead.
        shutil.copy2(src_dir / fname, link)

REQUIRED_COLS = {"writing_id", "l1", "cefr_level", "text"}
for fname in EXPECTED:
    head = pd.read_csv(Path("data") / fname, nrows=1)
    miss = REQUIRED_COLS - set(head.columns)
    status = "OK" if not miss else f"MISSING {sorted(miss)}"
    print(f"  {fname:<35}  {status}")

## Train each model in `MODELS`, then predict on the 3 external test sets

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

MODELS_ROOT = str(Path(OUTPUT_DIR) / "models")
PREDS_ROOT  = str(Path(OUTPUT_DIR) / "predictions")
Path(MODELS_ROOT).mkdir(parents=True, exist_ok=True)
Path(PREDS_ROOT).mkdir(parents=True, exist_ok=True)

TEST_FILES = [
    "data/norm-EFCAMDAT-test.csv",
    "data/norm-KUPA-KEYS.csv",
    "data/norm-CELVA-SP.csv",
]


def _stream(cmd: list[str]) -> str:
    """Run cmd, stream stdout+stderr live, return the last non-empty line.

    train.py and predict.py both print their output folder as the final line,
    so that line is what the caller wants.
    """
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    last = ""
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        if line.strip():
            last = line.strip()
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{cmd[0]} {cmd[1]} exited {proc.returncode}")
    return last


results: list[dict] = []
for model in MODELS:
    print(f"\n{'=' * 72}\n  TRAIN  {model}\n{'=' * 72}")
    model_dir = _stream([
        sys.executable, "train.py",
        "--config", "configs/default.yaml",
        "--model-name", model,
        "--output-root", MODELS_ROOT,
        "--predictions-root", PREDS_ROOT,
    ])

    print(f"\n{'=' * 72}\n  PREDICT  {model}\n{'=' * 72}")
    pred_dir = _stream([
        sys.executable, "predict.py",
        "--model", model_dir,
        "--test-files", *TEST_FILES,
        "--predictions-root", PREDS_ROOT,
    ])

    # Collate metrics per test file for the summary table.
    report = json.loads((Path(pred_dir) / "report.json").read_text(encoding="utf-8"))
    for test_name, entry in report["test_files"].items():
        m = entry["metrics"]
        results.append({
            "model":      model,
            "test_file":  test_name,
            "n":          m.get("n"),
            "accuracy":   m.get("accuracy"),
            "macro_f1":   m.get("macro_f1"),
            "qwk":        m.get("quadratic_weighted_kappa"),
            "adj_acc":    m.get("adjacent_accuracy"),
            "model_dir":  model_dir,
            "pred_dir":   pred_dir,
        })
print("\nAll done.")

## Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_colwidth = 60
display_cols = ["model", "test_file", "n", "accuracy", "macro_f1", "qwk", "adj_acc"]
df[display_cols]

In [ ]:
# Persist the summary alongside predictions so you can compare across runs.
from pathlib import Path

summary_path = Path(OUTPUT_DIR) / "summary.csv"
df.to_csv(summary_path, index=False)
print("Saved:", summary_path)